# Modern Portfolio Theory

## Objective

Modern Portfolio Theory (MPT), associated with Harry Markowitz, models a portfolio as a weighted combination of assets and studies the trade-off between expected return and risk.

This notebook develops the mathematical building blocks used later for portfolio optimization:

- Portfolio weights and the budget constraint
- Historical mean returns
- The covariance matrix
- Expected portfolio return
- Portfolio variance and volatility

We use daily adjusted-price returns for five Indian equities from 2020 through 2025. The statistics are daily, not annualized, and are estimates based on historical data rather than forecasts.

## Load and Prepare Data

We download adjusted historical prices, calculate daily percentage returns, and explicitly order the return columns to match the `stocks` list. Consistent ordering is essential because a weight vector's first element must correspond to the first return column, and so on.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

stocks = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "ICICIBANK.NS"]

data = yf.download(
    stocks,
    start="2020-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)

if data.empty:
    raise ValueError("No data was downloaded. Check the tickers, date range, or internet connection.")

close_prices = data["Close"].reindex(columns=stocks).dropna(how="all")
returns = close_prices.pct_change().dropna()

## Portfolio Weights

A portfolio weight \(w_i\) is the fraction of portfolio capital allocated to asset \(i\). For a fully invested portfolio,

$$
\sum_{i=1}^{n} w_i = 1.
$$

In this notebook we begin with an equal-weight portfolio. With five stocks, each receives 20% of the capital. Later, the weights will be treated as decision variables in optimization.

In [2]:
weights = np.full(len(stocks), 1 / len(stocks))
pd.Series(weights, index=stocks, name="Portfolio Weight")

,Portfolio Weight
RELIANCE.NS,0.2
TCS.NS,0.2
INFY.NS,0.2
HDFCBANK.NS,0.2
ICICIBANK.NS,0.2


Each stock receives the same allocation, and the weights sum to one. This is a baseline portfolio rather than an optimized allocation.

## Expected Returns

For each asset, the historical mean daily return estimates its expected daily return:

$$
\hat{\mu}_i = \frac{1}{N}\sum_{t=1}^{N}R_{i,t}.
$$

Here, $$(R_{i,t})$$ is the daily return of asset \(i\) on day \(t\), and \(N\) is the number of daily return observations. The vector of mean returns is denoted by $$(\hat{\mu}).$$

In [3]:
expected_returns = returns[stocks].mean()
expected_returns

,0
Ticker,
RELIANCE.NS,0.000726
TCS.NS,0.000464
INFY.NS,0.000779
HDFCBANK.NS,0.000463
ICICIBANK.NS,0.000816


## Covariance Matrix

The covariance matrix \(\Sigma\) describes how returns vary together:

$$
\Sigma =
\begin{bmatrix}
\operatorname{Var}(R_1) & \operatorname{Cov}(R_1,R_2) & \cdots \\
\operatorname{Cov}(R_2,R_1) & \operatorname{Var}(R_2) & \cdots \\
\vdots & \vdots & \ddots
\end{bmatrix}.
$$

Diagonal entries are individual return variances; off-diagonal entries are pairwise covariances. The covariance matrix is central to portfolio risk because assets' co-movement affects the risk of their combination.

In [4]:
cov_matrix = returns[stocks].cov()
cov_matrix

Ticker,RELIANCE.NS,TCS.NS,INFY.NS,HDFCBANK.NS,ICICIBANK.NS
Ticker,,,,,
RELIANCE.NS,0.000317,0.000101,0.000107,0.000127,0.000144
TCS.NS,0.000101,0.000224,0.000184,0.000073,0.000090
INFY.NS,0.000107,0.000184,0.000298,0.000096,0.000118
HDFCBANK.NS,0.000127,0.000073,0.000096,0.000262,0.000192
ICICIBANK.NS,0.000144,0.000090,0.000118,0.000192,0.000350


## Expected Portfolio Return

The expected return of a portfolio is the weighted sum of the expected returns of its assets:

$$
\hat{\mu}_p = w^T\hat{\mu}.
$$

This is a daily estimate because the input returns are daily. It assumes the weights refer to the same asset order as the return vector.

In [5]:
portfolio_return = float(np.dot(weights, expected_returns))
print(f"Expected daily portfolio return: {portfolio_return:.6f} ({portfolio_return:.3%})")

Expected daily portfolio return: 0.000650 (0.065%)


## Portfolio Variance

Portfolio variance accounts for both individual asset risk and the covariance between assets:

$$
\sigma_p^2 = w^T\Sigma w.
$$

A portfolio's variance is therefore not simply the weighted average of individual variances. The covariance terms determine how co-movement affects total risk.

In [6]:
portfolio_variance = float(weights.T @ cov_matrix.to_numpy() @ weights)
print(f"Portfolio daily variance: {portfolio_variance:.8f}")

Portfolio daily variance: 0.00015665


## Portfolio Volatility

Portfolio volatility is the square root of variance:

$$
\sigma_p = \sqrt{w^T\Sigma w}.
$$

It is the standard deviation of portfolio daily returns, so it is expressed in daily-return units.

In [7]:
portfolio_volatility = float(np.sqrt(portfolio_variance))
print(f"Portfolio daily volatility: {portfolio_volatility:.6f} ({portfolio_volatility:.2%})")

Portfolio daily volatility: 0.012516 (1.25%)


## Summary

The table collects the daily return and risk estimates for the equal-weight portfolio. The values should be interpreted together: return describes the historical average, variance is a squared risk measure, and volatility is the standard-deviation measure.

In [8]:
summary = pd.DataFrame({
    "Expected Daily Return": [portfolio_return],
    "Daily Variance": [portfolio_variance],
    "Daily Volatility": [portfolio_volatility]
}, index=["Equal-Weight Portfolio"])
summary

,Expected Daily Return,Daily Variance,Daily Volatility
Equal-Weight Portfolio,0.00065,0.000157,0.012516


# Conclusions

- Portfolio weights describe how capital is allocated and sum to one for a fully invested portfolio.
- Expected portfolio return is the weighted sum of asset mean returns.
- Portfolio variance depends on both individual variances and covariances.
- Portfolio volatility is the square root of portfolio variance.
- These quantities form the inputs and objective functions used in the following portfolio-optimization notebooks.

The estimates are historical and sensitive to the chosen assets and sample period.